# CSE 472 Assignment 2: Decision Trees, Random Forests, and Extra Trees

Implementation of tree-based classification algorithms from scratch.

In [27]:
# importing stuff we need
import numpy as np
from collections import Counter
from typing import Optional, Tuple, List, Union
import warnings
warnings.filterwarnings('ignore')  # ignore warnings

## Metrics

accuracy, f1 and auroc functions

In [28]:
# calculate accuracy
def accuracy_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    # just count how many are correct
    result = np.mean(y_true == y_pred)
    return result

In [29]:
def f1_score(y_true: np.ndarray, y_pred: np.ndarray, average: str = 'macro') -> float:
    # calculate f1 score
    # for each class we need precision and recall
    unique_labels = np.unique(np.concatenate([y_true, y_pred]))
    f1_scores = []  # store f1 for each class

    for label in unique_labels:
        # calculate TP, FP, FN
        tp = np.sum((y_true == label) & (y_pred == label))
        fp = np.sum((y_true != label) & (y_pred == label))
        fn = np.sum((y_true == label) & (y_pred != label))

        # precision = tp / (tp + fp)
        if (tp + fp) > 0:
            precision = tp / (tp + fp)
        else:
            precision = 0
            
        # recall = tp / (tp + fn)
        if (tp + fn) > 0:
            recall = tp / (tp + fn)
        else:
            recall = 0

        # f1 = 2 * precision * recall / (precision + recall)
        if (precision + recall) > 0:
            f1 = 2 * precision * recall / (precision + recall)
        else:
            f1 = 0
        f1_scores.append(f1)

    f1_scores = np.array(f1_scores)

    # return average
    if average == 'macro':
        return np.mean(f1_scores)
    elif average == 'weighted':
        weights = np.array([np.sum(y_true == label) for label in unique_labels])
        return np.sum(f1_scores * weights) / np.sum(weights)
    else:
        raise ValueError(f"Unknown average: {average}")

In [30]:
def auroc_score(y_true: np.ndarray, y_proba: np.ndarray) -> float:
    # calculate AUROC score
    # using one vs rest strategy
    unique_labels = np.unique(y_true)
    n_classes = len(unique_labels)
    auroc_scores = []

    for i, label in enumerate(unique_labels):
        # make it binary - current class vs rest
        y_binary = (y_true == label).astype(int)
        y_scores = y_proba[:, i]

        # sort by scores
        indices = np.argsort(y_scores)[::-1]  # descending
        y_binary_sorted = y_binary[indices]

        # count positives and negatives
        n_pos = np.sum(y_binary)
        n_neg = len(y_binary) - n_pos

        if n_pos == 0 or n_neg == 0:
            auroc_scores.append(0.5)
            continue

        # calculate AUC
        tpr = 0.0  # true positive rate
        fpr = 0.0  # false positive rate
        auc = 0.0
        prev_fpr = 0.0
        
        for j in range(len(y_binary_sorted)):
            if y_binary_sorted[j] == 1:
                # true positive
                tpr += 1.0 / n_pos
            else:
                # false positive
                prev_fpr = fpr
                fpr += 1.0 / n_neg
                auc += (fpr - prev_fpr) * tpr  # add area

        auroc_scores.append(auc)

    # return average
    return np.mean(auroc_scores)

## Impurity stuff

gini and entropy

In [31]:
# gini impurity
def gini_impurity(y: np.ndarray) -> float:
    if len(y) == 0:
        return 0
    y_int = y.astype(int)
    if np.any(y_int < 0):
        raise ValueError("Labels must be non-negative integers")
    counts = np.bincount(y_int)
    probs = counts / len(y)
    gini = 1 - np.sum(probs ** 2)  # formula for gini
    return gini


# entropy
def entropy(y: np.ndarray) -> float:
    if len(y) == 0:
        return 0
    y_int = y.astype(int)
    if np.any(y_int < 0):
        raise ValueError("Labels must be non-negative integers")
    counts = np.bincount(y_int)
    probs = counts[counts > 0] / len(y)
    ent = -np.sum(probs * np.log2(probs + 1e-10))  # entropy formula
    return ent

## Tree Node

node class for trees

In [32]:
class DecisionTreeNode:
    # node for decision tree
    def __init__(self, feature_idx: int = None, threshold: float = None,
                 left: 'DecisionTreeNode' = None, right: 'DecisionTreeNode' = None,
                 value: Union[int, np.ndarray] = None, is_leaf: bool = False):
        self.feature_idx = feature_idx  # which feature to split on
        self.threshold = threshold  # threshold value
        self.left = left  # left child
        self.right = right  # right child
        self.value = value  # class label if leaf
        self.is_leaf = is_leaf  # is it a leaf?

## Decision Tree

main decision tree class

In [33]:
class DecisionTreeClassifier:
    # decision tree classifier
    def __init__(self, max_depth: Optional[int] = None,
                 min_samples_split: int = 2,
                 criterion: str = 'gini',
                 random_state: Optional[int] = None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.criterion = criterion
        self.random_state = random_state
        self.root = None
        self.n_classes = None
        self.n_features = None

        if random_state is not None:
            np.random.seed(random_state)

    def _get_criterion_func(self):
        # get the criterion function to use
        if self.criterion == 'gini':
            return gini_impurity
        elif self.criterion == 'entropy':
            return entropy
        else:
            raise ValueError(f"Unknown criterion: {self.criterion}")

    def _find_best_split(self, X: np.ndarray, y: np.ndarray,
                         feature_indices: List[int] = None) -> Tuple[int, float]:
        # find best split
        criterion_func = self._get_criterion_func()
        n_samples, n_features = X.shape

        if feature_indices is None:
            feature_indices = list(range(n_features))

        best_gain = -np.inf
        best_feature = None
        best_threshold = None

        parent_impurity = criterion_func(y)

        # try all features
        for feat_idx in feature_indices:
            feature_values = X[:, feat_idx]
            unique_values = np.unique(feature_values)

            if len(unique_values) <= 1:
                continue

            # try all thresholds
            thresholds = (unique_values[:-1] + unique_values[1:]) / 2

            for threshold in thresholds:
                left_mask = feature_values <= threshold
                right_mask = ~left_mask

                if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
                    continue

                n_left = np.sum(left_mask)
                n_right = np.sum(right_mask)

                # calculate impurity
                left_impurity = criterion_func(y[left_mask])
                right_impurity = criterion_func(y[right_mask])

                weighted_impurity = (n_left / n_samples) * left_impurity + \
                                    (n_right / n_samples) * right_impurity

                # information gain
                gain = parent_impurity - weighted_impurity

                # update best
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feat_idx
                    best_threshold = threshold

        return best_feature, best_threshold

    def _build_tree(self, X: np.ndarray, y: np.ndarray,
                    depth: int = 0) -> DecisionTreeNode:
        # build tree recursively
        n_samples = X.shape[0]

        # check stopping conditions
        if (self.max_depth is not None and depth >= self.max_depth) or \
           n_samples < self.min_samples_split or \
           len(np.unique(y)) == 1:
            leaf_value = self._get_leaf_value(y)
            return DecisionTreeNode(value=leaf_value, is_leaf=True)

        # find best split
        best_feature, best_threshold = self._find_best_split(X, y)

        if best_feature is None:
            leaf_value = self._get_leaf_value(y)
            return DecisionTreeNode(value=leaf_value, is_leaf=True)

        # split data
        left_mask = X[:, best_feature] <= best_threshold
        right_mask = ~left_mask

        # build left and right subtrees
        left_child = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right_child = self._build_tree(X[right_mask], y[right_mask], depth + 1)

        return DecisionTreeNode(
            feature_idx=best_feature,
            threshold=best_threshold,
            left=left_child,
            right=right_child,
            is_leaf=False
        )

    def _get_leaf_value(self, y: np.ndarray) -> int:
        # get majority class
        if len(y) == 0:
            return 0
        counts = np.bincount(y.astype(int))
        return int(np.argmax(counts))

    def fit(self, X: np.ndarray, y: np.ndarray) -> 'DecisionTreeClassifier':
        # fit the tree
        X = np.asarray(X)
        y = np.asarray(y)

        self.n_classes = len(np.unique(y))
        self.n_features = X.shape[1]

        if self.random_state is not None:
            np.random.seed(self.random_state)

        self.root = self._build_tree(X, y)
        return self

    def _predict_single(self, x: np.ndarray, node: DecisionTreeNode) -> int:
        # predict for one sample
        if node.is_leaf:
            return node.value

        # go left or right
        if x[node.feature_idx] <= node.threshold:
            return self._predict_single(x, node.left)
        else:
            return self._predict_single(x, node.right)

    def predict(self, X: np.ndarray) -> np.ndarray:
        # predict for all samples
        X = np.asarray(X)
        predictions = []
        for x in X:
            pred = self._predict_single(x, self.root)
            predictions.append(pred)
        return np.array(predictions)

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        # predict probabilities
        X = np.asarray(X)
        proba = []

        for x in X:
            node = self.root
            # traverse to leaf
            while not node.is_leaf:
                if x[node.feature_idx] <= node.threshold:
                    node = node.left
                else:
                    node = node.right

            # create probability array
            p = np.zeros(self.n_classes)
            p[node.value] = 1.0
            proba.append(p)

        return np.array(proba)

## Random Forest Tree

special tree for random forest

In [34]:
class _RandomForestTree(DecisionTreeClassifier):
    # tree for random forest with random features
    
    def __init__(self, max_depth, min_samples_split, criterion, random_state, n_features_split, rng):
        super().__init__(max_depth=max_depth, min_samples_split=min_samples_split,
                         criterion=criterion, random_state=random_state)
        self.n_features_split = n_features_split
        self._rng = rng
    
    def _find_best_split(self, X: np.ndarray, y: np.ndarray,
                         feature_indices: List[int] = None) -> Tuple[int, float]:
        # find best split using random features
        criterion_func = self._get_criterion_func()
        n_samples, n_total_features = X.shape
        
        # select random features
        feature_indices = self._rng.choice(n_total_features, 
                                          size=min(self.n_features_split, n_total_features), 
                                          replace=False)

        best_gain = -np.inf
        best_feature = None
        best_threshold = None
        parent_impurity = criterion_func(y)

        # try each feature
        for feat_idx in feature_indices:
            feature_values = X[:, feat_idx]
            unique_values = np.unique(feature_values)

            if len(unique_values) <= 1:
                continue

            thresholds = (unique_values[:-1] + unique_values[1:]) / 2

            # try each threshold
            for threshold in thresholds:
                left_mask = feature_values <= threshold
                right_mask = ~left_mask

                if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
                    continue

                n_left = np.sum(left_mask)
                n_right = np.sum(right_mask)

                left_impurity = criterion_func(y[left_mask])
                right_impurity = criterion_func(y[right_mask])

                weighted_impurity = (n_left / n_samples) * left_impurity + \
                                    (n_right / n_samples) * right_impurity

                gain = parent_impurity - weighted_impurity

                if gain > best_gain:
                    best_gain = gain
                    best_feature = feat_idx
                    best_threshold = threshold

        return best_feature, best_threshold

## Random Forest

ensemble of trees

In [35]:
class RandomForestClassifier:
    # random forest classifier
    
    def __init__(self, n_estimators: int = 100,
                 max_depth: Optional[int] = None,
                 min_samples_split: int = 2,
                 max_features: Union[str, int, float] = 'sqrt',
                 criterion: str = 'gini',
                 random_state: Optional[int] = None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.criterion = criterion
        self.random_state = random_state
        self.trees = []
        self.n_classes = None
        self._rng = np.random.RandomState(random_state)

    def _get_n_features(self, n_features: int) -> int:
        # figure out how many features to use
        if isinstance(self.max_features, str):
            if self.max_features == 'sqrt':
                return int(np.sqrt(n_features))
            elif self.max_features == 'log2':
                return int(np.log2(n_features))
            else:
                raise ValueError(f"Unknown max_features: {self.max_features}")
        elif isinstance(self.max_features, float):
            return int(self.max_features * n_features)
        elif isinstance(self.max_features, int):
            return self.max_features
        else:
            return n_features

    def _bootstrap_sample(self, X: np.ndarray, y: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        # create bootstrap sample
        n_samples = X.shape[0]
        indices = self._rng.choice(n_samples, size=n_samples, replace=True)
        return X[indices], y[indices]

    def fit(self, X: np.ndarray, y: np.ndarray) -> 'RandomForestClassifier':
        # fit random forest
        X = np.asarray(X)
        y = np.asarray(y)

        self.n_classes = len(np.unique(y))
        n_features = X.shape[1]
        n_features_split = self._get_n_features(n_features)

        self.trees = []

        # build each tree
        for i in range(self.n_estimators):
            # bootstrap sample
            X_boot, y_boot = self._bootstrap_sample(X, y)
            tree_rng = np.random.RandomState(self.random_state + i if self.random_state is not None else None)
            tree = _RandomForestTree(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                criterion=self.criterion,
                random_state=self.random_state + i if self.random_state is not None else None,
                n_features_split=n_features_split,
                rng=tree_rng
            )

            tree.fit(X_boot, y_boot)
            self.trees.append(tree)

        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        # predict using majority voting
        X = np.asarray(X)
        predictions = np.array([tree.predict(X) for tree in self.trees])

        result = []
        # for each sample
        for i in range(X.shape[0]):
            votes = predictions[:, i]
            # majority vote
            result.append(int(np.bincount(votes.astype(int)).argmax()))

        return np.array(result)

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        # predict probabilities
        X = np.asarray(X)
        probas = np.array([tree.predict_proba(X) for tree in self.trees])
        # average probabilities
        return np.mean(probas, axis=0)

## Extra Trees

extra trees with random thresholds

In [36]:
class ExtraTreesClassifier:
    # extra trees classifier
    
    def __init__(self, n_estimators: int = 100,
                 max_depth: Optional[int] = None,
                 min_samples_split: int = 2,
                 max_features: Union[str, int, float] = 'sqrt',
                 criterion: str = 'gini',
                 random_state: Optional[int] = None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.criterion = criterion
        self.random_state = random_state
        self.trees = []
        self.n_classes = None
        self._rng = np.random.RandomState(random_state)

    def _get_n_features(self, n_features: int) -> int:
        # calculate number of features
        if isinstance(self.max_features, str):
            if self.max_features == 'sqrt':
                return int(np.sqrt(n_features))
            elif self.max_features == 'log2':
                return int(np.log2(n_features))
            else:
                raise ValueError(f"Unknown max_features: {self.max_features}")
        elif isinstance(self.max_features, float):
            return int(self.max_features * n_features)
        elif isinstance(self.max_features, int):
            return self.max_features
        else:
            return n_features

    def _extra_randomized_split(self, X: np.ndarray, y: np.ndarray,
                                 n_features: int, criterion_func) -> Tuple[int, float]:
        # find split with random thresholds (extra trees special feature!)
        n_samples, n_total_features = X.shape
        feature_indices = self._rng.choice(n_total_features, size=min(n_features, n_total_features), replace=False)

        best_gain = -np.inf
        best_feature = None
        best_threshold = None
        parent_impurity = criterion_func(y)

        # try random features
        for feat_idx in feature_indices:
            feature_values = X[:, feat_idx]
            min_val, max_val = feature_values.min(), feature_values.max()

            if min_val == max_val:
                continue

            # RANDOM threshold!! this is what makes it extra trees
            threshold = self._rng.uniform(min_val, max_val)

            left_mask = feature_values <= threshold
            right_mask = ~left_mask

            if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
                continue

            n_left = np.sum(left_mask)
            n_right = np.sum(right_mask)

            left_impurity = criterion_func(y[left_mask])
            right_impurity = criterion_func(y[right_mask])

            weighted_impurity = (n_left / n_samples) * left_impurity + \
                                (n_right / n_samples) * right_impurity

            gain = parent_impurity - weighted_impurity

            if gain > best_gain:
                best_gain = gain
                best_feature = feat_idx
                best_threshold = threshold

        return best_feature, best_threshold

    def _build_extra_tree(self, X: np.ndarray, y: np.ndarray,
                          depth: int, n_features_split: int, criterion_func) -> DecisionTreeNode:
        # build one extra tree
        n_samples = X.shape[0]

        # stopping conditions
        if (self.max_depth is not None and depth >= self.max_depth) or \
           n_samples < self.min_samples_split or \
           len(np.unique(y)) == 1:
            leaf_value = self._get_leaf_value(y)
            return DecisionTreeNode(value=leaf_value, is_leaf=True)

        # find best split
        best_feature, best_threshold = self._extra_randomized_split(
            X, y, n_features_split, criterion_func
        )

        if best_feature is None:
            leaf_value = self._get_leaf_value(y)
            return DecisionTreeNode(value=leaf_value, is_leaf=True)

        # split
        left_mask = X[:, best_feature] <= best_threshold
        right_mask = ~left_mask

        # recursively build children
        left_child = self._build_extra_tree(X[left_mask], y[left_mask], depth + 1, n_features_split, criterion_func)
        right_child = self._build_extra_tree(X[right_mask], y[right_mask], depth + 1, n_features_split, criterion_func)

        return DecisionTreeNode(
            feature_idx=best_feature,
            threshold=best_threshold,
            left=left_child,
            right=right_child,
            is_leaf=False
        )

    def _get_leaf_value(self, y: np.ndarray) -> int:
        # get majority class for leaf
        if len(y) == 0:
            return 0
        counts = np.bincount(y.astype(int))
        return int(np.argmax(counts))

    def fit(self, X: np.ndarray, y: np.ndarray) -> 'ExtraTreesClassifier':
        # fit extra trees
        X = np.asarray(X)
        y = np.asarray(y)

        self.n_classes = len(np.unique(y))
        n_features = X.shape[1]
        n_features_split = self._get_n_features(n_features)
        criterion_func = gini_impurity if self.criterion == 'gini' else entropy

        self.trees = []

        # build trees
        for i in range(self.n_estimators):
            if self.random_state is not None:
                self._rng = np.random.RandomState(self.random_state + i)
            
            tree = self._build_extra_tree(X, y, depth=0, n_features_split=n_features_split, criterion_func=criterion_func)
            self.trees.append(tree)

        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        # predict using voting
        X = np.asarray(X)
        predictions = []

        for x in X:
            votes = []
            # get prediction from each tree
            for tree in self.trees:
                node = tree
                while not node.is_leaf:
                    if x[node.feature_idx] <= node.threshold:
                        node = node.left
                    else:
                        node = node.right
                votes.append(node.value)

            # majority vote
            predictions.append(int(np.bincount(votes).argmax()))

        return np.array(predictions)

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        # predict probabilities
        X = np.asarray(X)
        probas = []

        for x in X:
            votes = []
            for tree in self.trees:
                node = tree
                while not node.is_leaf:
                    if x[node.feature_idx] <= node.threshold:
                        node = node.left
                    else:
                        node = node.right
                votes.append(node.value)

            # convert votes to probabilities
            p = np.zeros(self.n_classes)
            for v in votes:
                p[v] += 1
            p = p / len(votes)
            probas.append(p)

        return np.array(probas)

## Helper Functions

some utility functions

In [37]:
def print_dataset_statistics(dataset_name: str, X: np.ndarray, y: np.ndarray):
    # print stats about dataset
    n_samples, n_features = X.shape
    unique_classes, class_counts = np.unique(y, return_counts=True)
    n_classes = len(unique_classes)
    
    print(f"\n{'='*80}")
    print(f"Dataset Statistics: {dataset_name}")
    print(f"{'='*80}")
    print(f"\nBasic Information:")
    print(f"  - Number of samples: {n_samples}")
    print(f"  - Number of features: {n_features}")
    print(f"  - Number of classes: {n_classes}")
    
    print(f"\nClass Distribution:")
    for cls, count in zip(unique_classes, class_counts):
        percentage = (count / n_samples) * 100
        print(f"  - Class {int(cls)}: {count} samples ({percentage:.2f}%)")
    
    print(f"\nFeature Statistics:")
    print(f"  - Feature means: {np.mean(X, axis=0)}")
    print(f"  - Feature std devs: {np.std(X, axis=0)}")
    print(f"  - Feature mins: {np.min(X, axis=0)}")
    print(f"  - Feature maxs: {np.max(X, axis=0)}")
    
    balance_ratio = np.min(class_counts) / np.max(class_counts)
    print(f"\nClass Balance:")
    print(f"  - Balance ratio (min/max): {balance_ratio:.3f}")
    if balance_ratio < 0.5:
        print(f"  - WARNING: Dataset is imbalanced!")
    else:
        print(f"  - Dataset is reasonably balanced")
    
    print(f"{'='*80}\n")

In [38]:
def evaluate_model(model, X_train: np.ndarray, y_train: np.ndarray,
                   X_test: np.ndarray, y_test: np.ndarray) -> dict:
    # evaluate model and return metrics
    model.fit(X_train, y_train)

    # get predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # get probabilities
    y_train_proba = model.predict_proba(X_train)
    y_test_proba = model.predict_proba(X_test)

    # calculate all metrics
    results = {
        'train_accuracy': accuracy_score(y_train, y_train_pred),
        'test_accuracy': accuracy_score(y_test, y_test_pred),
        'train_f1': f1_score(y_train, y_train_pred),
        'test_f1': f1_score(y_test, y_test_pred),
        'train_auroc': auroc_score(y_train, y_train_proba),
        'test_auroc': auroc_score(y_test, y_test_proba),
    }

    return results

In [39]:
def print_results_table(dataset_name: str, custom_results: dict, sklearn_results: dict):
    # print results in table format
    print(f"\n{'='*80}")
    print(f"Results for {dataset_name} Dataset")
    print(f"{'='*80}")
    print(f"\nCustom Implementations:")
    print(f"{'-'*80}")
    print(f"{'Model':<20} {'Train Acc':<12} {'Test Acc':<12} {'Train F1':<12} {'Test F1':<12} {'Test AUROC':<12}")
    print(f"{'-'*80}")

    for model_name, results in custom_results.items():
        print(f"{model_name:<20} {results['train_accuracy']:<12.4f} {results['test_accuracy']:<12.4f} "
              f"{results['train_f1']:<12.4f} {results['test_f1']:<12.4f} {results['test_auroc']:<12.4f}")

    print(f"\nscikit-learn Implementations:")
    print(f"{'-'*80}")
    print(f"{'Model':<20} {'Train Acc':<12} {'Test Acc':<12} {'Train F1':<12} {'Test F1':<12} {'Test AUROC':<12}")
    print(f"{'-'*80}")

    for model_name, results in sklearn_results.items():
        print(f"{model_name:<20} {results['train_accuracy']:<12.4f} {results['test_accuracy']:<12.4f} "
              f"{results['train_f1']:<12.4f} {results['test_f1']:<12.4f} {results['test_auroc']:<12.4f}")

    print(f"{'='*80}\n")

## Experiments

running experiments on datasets

In [40]:
# import sklearn stuff
from sklearn.datasets import load_iris, load_wine
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier as SklearnDecisionTree
from sklearn.ensemble import RandomForestClassifier as SklearnRandomForest
from sklearn.ensemble import ExtraTreesClassifier as SklearnExtraTrees

# random state
RANDOM_STATE = 42

print("\n" + "="*80)
print("CSE 472 Assignment 2: Decision Trees, Random Forests, and Extra Trees")
print("="*80)
print("\nRunning experiments with random_state = 42...")


CSE 472 Assignment 2: Decision Trees, Random Forests, and Extra Trees

Running experiments with random_state = 42...


In [41]:
# hyperparameters
hyperparams = {
    'max_depth': 10,
    'min_samples_split': 2,
    'n_estimators': 50,
    'max_features': 'sqrt',
    'random_state': RANDOM_STATE
}

### Iris Dataset

In [42]:
print("\n[1/2] Loading Iris dataset...")
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

print_dataset_statistics("Iris", X_iris, y_iris)

# split data
X_train, X_test, y_train, y_test = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=RANDOM_STATE, stratify=y_iris
)

print(f"\nTrain/Test Split:")
print(f"  - Training samples: {len(y_train)}")
print(f"  - Test samples: {len(y_test)}")


[1/2] Loading Iris dataset...

Dataset Statistics: Iris

Basic Information:
  - Number of samples: 150
  - Number of features: 4
  - Number of classes: 3

Class Distribution:
  - Class 0: 50 samples (33.33%)
  - Class 1: 50 samples (33.33%)
  - Class 2: 50 samples (33.33%)

Feature Statistics:
  - Feature means: [5.84333333 3.05733333 3.758      1.19933333]
  - Feature std devs: [0.82530129 0.43441097 1.75940407 0.75969263]
  - Feature mins: [4.3 2.  1.  0.1]
  - Feature maxs: [7.9 4.4 6.9 2.5]

Class Balance:
  - Balance ratio (min/max): 1.000
  - Dataset is reasonably balanced


Train/Test Split:
  - Training samples: 105
  - Test samples: 45


In [43]:
# train custom models on iris
print("  - Training Custom Decision Tree...")
custom_dt_iris = evaluate_model(
    DecisionTreeClassifier(max_depth=hyperparams['max_depth'],
                           min_samples_split=hyperparams['min_samples_split'],
                           random_state=hyperparams['random_state']),
    X_train, y_train, X_test, y_test
)

print("  - Training Custom Random Forest...")
custom_rf_iris = evaluate_model(
    RandomForestClassifier(n_estimators=hyperparams['n_estimators'],
                           max_depth=hyperparams['max_depth'],
                           min_samples_split=hyperparams['min_samples_split'],
                           max_features=hyperparams['max_features'],
                           random_state=hyperparams['random_state']),
    X_train, y_train, X_test, y_test
)

print("  - Training Custom Extra Trees...")
custom_et_iris = evaluate_model(
    ExtraTreesClassifier(n_estimators=hyperparams['n_estimators'],
                         max_depth=hyperparams['max_depth'],
                         min_samples_split=hyperparams['min_samples_split'],
                         max_features=hyperparams['max_features'],
                         random_state=hyperparams['random_state']),
    X_train, y_train, X_test, y_test
)

  - Training Custom Decision Tree...
  - Training Custom Random Forest...
  - Training Custom Extra Trees...
  - Training Custom Extra Trees...


In [44]:
# train sklearn models on iris
print("  - Training sklearn Decision Tree...")
sklearn_dt_iris = evaluate_model(
    SklearnDecisionTree(max_depth=hyperparams['max_depth'],
                        min_samples_split=hyperparams['min_samples_split'],
                        random_state=hyperparams['random_state']),
    X_train, y_train, X_test, y_test
)

print("  - Training sklearn Random Forest...")
sklearn_rf_iris = evaluate_model(
    SklearnRandomForest(n_estimators=hyperparams['n_estimators'],
                        max_depth=hyperparams['max_depth'],
                        min_samples_split=hyperparams['min_samples_split'],
                        max_features=hyperparams['max_features'],
                        random_state=hyperparams['random_state']),
    X_train, y_train, X_test, y_test
)

print("  - Training sklearn Extra Trees...")
sklearn_et_iris = evaluate_model(
    SklearnExtraTrees(n_estimators=hyperparams['n_estimators'],
                      max_depth=hyperparams['max_depth'],
                      min_samples_split=hyperparams['min_samples_split'],
                      max_features=hyperparams['max_features'],
                      random_state=hyperparams['random_state']),
    X_train, y_train, X_test, y_test
)

  - Training sklearn Decision Tree...
  - Training sklearn Random Forest...
  - Training sklearn Extra Trees...


In [45]:
# print iris results
print_results_table(
    "Iris",
    {
        'Decision Tree': custom_dt_iris,
        'Random Forest': custom_rf_iris,
        'Extra Trees': custom_et_iris
    },
    {
        'Decision Tree': sklearn_dt_iris,
        'Random Forest': sklearn_rf_iris,
        'Extra Trees': sklearn_et_iris
    }
)


Results for Iris Dataset

Custom Implementations:
--------------------------------------------------------------------------------
Model                Train Acc    Test Acc     Train F1     Test F1      Test AUROC  
--------------------------------------------------------------------------------
Decision Tree        1.0000       0.9111       1.0000       0.9111       0.9274      
Random Forest        1.0000       0.8889       1.0000       0.8878       0.9926      
Extra Trees          1.0000       0.9111       1.0000       0.9107       0.9941      

scikit-learn Implementations:
--------------------------------------------------------------------------------
Model                Train Acc    Test Acc     Train F1     Test F1      Test AUROC  
--------------------------------------------------------------------------------
Decision Tree        1.0000       0.9333       1.0000       0.9327       0.9541      
Random Forest        1.0000       0.8889       1.0000       0.8878       0.988

### Wine Dataset

In [46]:
print("\n[2/2] Loading Wine dataset...")
wine = load_wine()
X_wine, y_wine = wine.data, wine.target

print_dataset_statistics("Wine", X_wine, y_wine)

# split data
X_train, X_test, y_train, y_test = train_test_split(
    X_wine, y_wine, test_size=0.3, random_state=RANDOM_STATE, stratify=y_wine
)

print(f"\nTrain/Test Split:")
print(f"  - Training samples: {len(y_train)}")
print(f"  - Test samples: {len(y_test)}")


[2/2] Loading Wine dataset...

Dataset Statistics: Wine

Basic Information:
  - Number of samples: 178
  - Number of features: 13
  - Number of classes: 3

Class Distribution:
  - Class 0: 59 samples (33.15%)
  - Class 1: 71 samples (39.89%)
  - Class 2: 48 samples (26.97%)

Feature Statistics:
  - Feature means: [1.30006180e+01 2.33634831e+00 2.36651685e+00 1.94949438e+01
 9.97415730e+01 2.29511236e+00 2.02926966e+00 3.61853933e-01
 1.59089888e+00 5.05808988e+00 9.57449438e-01 2.61168539e+00
 7.46893258e+02]
  - Feature std devs: [8.09542915e-01 1.11400363e+00 2.73572294e-01 3.33016976e+00
 1.42423077e+01 6.24090564e-01 9.96048950e-01 1.24103260e-01
 5.70748849e-01 2.31176466e+00 2.27928607e-01 7.07993265e-01
 3.14021657e+02]
  - Feature mins: [1.103e+01 7.400e-01 1.360e+00 1.060e+01 7.000e+01 9.800e-01 3.400e-01
 1.300e-01 4.100e-01 1.280e+00 4.800e-01 1.270e+00 2.780e+02]
  - Feature maxs: [1.483e+01 5.800e+00 3.230e+00 3.000e+01 1.620e+02 3.880e+00 5.080e+00
 6.600e-01 3.580e+00 1

In [47]:
# train custom models on wine
print("  - Training Custom Decision Tree...")
custom_dt_wine = evaluate_model(
    DecisionTreeClassifier(max_depth=hyperparams['max_depth'],
                           min_samples_split=hyperparams['min_samples_split'],
                           random_state=hyperparams['random_state']),
    X_train, y_train, X_test, y_test
)

print("  - Training Custom Random Forest...")
custom_rf_wine = evaluate_model(
    RandomForestClassifier(n_estimators=hyperparams['n_estimators'],
                           max_depth=hyperparams['max_depth'],
                           min_samples_split=hyperparams['min_samples_split'],
                           max_features=hyperparams['max_features'],
                           random_state=hyperparams['random_state']),
    X_train, y_train, X_test, y_test
)

print("  - Training Custom Extra Trees...")
custom_et_wine = evaluate_model(
    ExtraTreesClassifier(n_estimators=hyperparams['n_estimators'],
                         max_depth=hyperparams['max_depth'],
                         min_samples_split=hyperparams['min_samples_split'],
                         max_features=hyperparams['max_features'],
                         random_state=hyperparams['random_state']),
    X_train, y_train, X_test, y_test
)

  - Training Custom Decision Tree...
  - Training Custom Random Forest...
  - Training Custom Extra Trees...
  - Training Custom Extra Trees...


In [48]:
# train sklearn models on wine
print("  - Training sklearn Decision Tree...")
sklearn_dt_wine = evaluate_model(
    SklearnDecisionTree(max_depth=hyperparams['max_depth'],
                        min_samples_split=hyperparams['min_samples_split'],
                        random_state=hyperparams['random_state']),
    X_train, y_train, X_test, y_test
)

print("  - Training sklearn Random Forest...")
sklearn_rf_wine = evaluate_model(
    SklearnRandomForest(n_estimators=hyperparams['n_estimators'],
                        max_depth=hyperparams['max_depth'],
                        min_samples_split=hyperparams['min_samples_split'],
                        max_features=hyperparams['max_features'],
                        random_state=hyperparams['random_state']),
    X_train, y_train, X_test, y_test
)

print("  - Training sklearn Extra Trees...")
sklearn_et_wine = evaluate_model(
    SklearnExtraTrees(n_estimators=hyperparams['n_estimators'],
                      max_depth=hyperparams['max_depth'],
                      min_samples_split=hyperparams['min_samples_split'],
                      max_features=hyperparams['max_features'],
                      random_state=hyperparams['random_state']),
    X_train, y_train, X_test, y_test
)

  - Training sklearn Decision Tree...
  - Training sklearn Random Forest...
  - Training sklearn Extra Trees...


In [49]:
# print wine results
print_results_table(
    "Wine",
    {
        'Decision Tree': custom_dt_wine,
        'Random Forest': custom_rf_wine,
        'Extra Trees': custom_et_wine
    },
    {
        'Decision Tree': sklearn_dt_wine,
        'Random Forest': sklearn_rf_wine,
        'Extra Trees': sklearn_et_wine
    }
)


Results for Wine Dataset

Custom Implementations:
--------------------------------------------------------------------------------
Model                Train Acc    Test Acc     Train F1     Test F1      Test AUROC  
--------------------------------------------------------------------------------
Decision Tree        1.0000       0.9815       1.0000       0.9827       0.9777      
Random Forest        1.0000       1.0000       1.0000       1.0000       1.0000      
Extra Trees          1.0000       1.0000       1.0000       1.0000       1.0000      

scikit-learn Implementations:
--------------------------------------------------------------------------------
Model                Train Acc    Test Acc     Train F1     Test F1      Test AUROC  
--------------------------------------------------------------------------------
Decision Tree        1.0000       0.9630       1.0000       0.9638       0.9636      
Random Forest        1.0000       1.0000       1.0000       1.0000       1.000

## Summary

final thoughts

In [50]:
# summary
print("\n" + "="*80)
print("SUMMARY: Custom vs scikit-learn Performance Comparison")
print("="*80)
print("\nKey Observations:")
print("1. Ensemble methods (RF, Extra Trees) generally outperform single Decision Trees")
print("2. Extra Trees often achieve similar accuracy with more randomness (faster training)")
print("3. Custom implementations show competitive results with sklearn")
print("="*80 + "\n")


SUMMARY: Custom vs scikit-learn Performance Comparison

Key Observations:
1. Ensemble methods (RF, Extra Trees) generally outperform single Decision Trees
2. Extra Trees often achieve similar accuracy with more randomness (faster training)
3. Custom implementations show competitive results with sklearn

